In [ ]:
import torch
import os 
import json
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR100
from torchvision import transforms, models

: 

: 

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
transform_train_vit = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_test_vit = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. Khởi tạo Dataset gốc với transform cho ViT
train_dataset_vit = CIFAR100(root="data", train=True, download=True, transform=transform_train_vit)
val_dataset_vit_clean = CIFAR100(root="data", train=True, download=False, transform=transform_test_vit)
test_dataset_vit = CIFAR100(root="data", train=False, download=True, transform=transform_test_vit)

# 3. Sử dụng đúng chỉ số phân chia (90% train, 10% val) đã cố định seed=42 từ trước
total_samples = len(train_dataset_vit)
train_size = int(0.9 * total_samples)

generator = torch.Generator().manual_seed(42)
permuted_indices = torch.randperm(total_samples, generator=generator).tolist()

train_indices = permuted_indices[:train_size]
val_indices = permuted_indices[train_size:]

# 4. Tạo Subset và DataLoader
TRAIN_SUBSET_VIT = Subset(train_dataset_vit, train_indices)
VAL_SUBSET_VIT = Subset(val_dataset_vit_clean, val_indices)

BATCH_SIZE = 32  # ViT tốn nhiều VRAM hơn ResNet, nên giảm batch size xuống 64 (hoặc 32 nếu GPU yếu)

train_loader = DataLoader(TRAIN_SUBSET_VIT, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(VAL_SUBSET_VIT, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset_vit, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [12]:
import timm
# Sử dụng vit_base_patch16_224 với pretrained=False để huấn luyện hoàn toàn từ đầu
vit_scratch = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=100)
vit_scratch = vit_scratch.to(device)

print("Đã khởi tạo thành công mô hình ViT-Base From Scratch.")

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 7.66 GiB of which 12.81 MiB is free. Including non-PyTorch memory, this process has 7.64 GiB memory in use. Of the allocated memory 7.32 GiB is allocated by PyTorch, and 162.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [5]:
EPOCHS = 50
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    vit_scratch.parameters(), 
    lr=0.001,
    betas=(0.9, 0.999), 
    weight_decay=0.01
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [6]:
import time
import copy
import psutil 

def train_model_with_metrics(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=50, device="cuda"):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    # Dictionary lưu lịch sử huấn luyện chi tiết
    history = {
        'train_loss': [], 'train_acc': [], 
        'val_loss': [], 'val_acc': [],
        'epoch_times': [],
        'gpu_max_memory_allocated_MB': [],
        'system_ram_usage_percent': []
    }

    # Reset thống kê peak memory của GPU trước khi train
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)

    total_start_time = time.time()

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # --- 1. Phase Huấn luyện (Training) ---
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Cập nhật learning rate theo scheduler sau mỗi epoch
        if scheduler is not None:
            scheduler.step()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = running_corrects.double() / len(train_loader.dataset)

        # --- 2. Phase Đánh giá (Validation) ---
        model.eval()
        val_loss = 0.0
        val_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = val_corrects.double() / len(val_loader.dataset)

        # --- 3. Ghi nhận thời gian và tài nguyên ---
        epoch_duration = time.time() - epoch_start_time
        
        # Đo lượng VRAM GPU đã sử dụng (Peak tính đến thời điểm hiện tại trong epoch)
        if device.type == 'cuda':
            gpu_mem_alloc = torch.cuda.max_memory_allocated(device) / (1024 * 1024) # Đổi sang MB
        else:
            gpu_mem_alloc = 0.0

        # Đo dung lượng RAM hệ thống đang dùng (%)
        sys_ram = psutil.virtual_memory().percent

        # Lưu vào history
        history['epoch_times'].append(epoch_duration)
        history['gpu_max_memory_allocated_MB'].append(gpu_mem_alloc)
        history['system_ram_usage_percent'].append(sys_ram)
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc.item())
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc.item())

        print(f'Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f}')
        print(f'Val   Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}')
        print(f'Time: {epoch_duration:.2f}s | GPU Mem Peak: {gpu_mem_alloc:.2f} MB | System RAM: {sys_ram}%\n')

        # Lưu lại trọng số tốt nhất dựa trên Validation Accuracy
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), 'vit_from_scratch.pth')

    total_duration = time.time() - total_start_time
    print(f'Training Completed in {total_duration // 60:.0f}m {total_duration % 60:.0f}s')
    print(f'Best Val Accuracy: {best_acc:.4f}')
    
    # Load lại trọng số tốt nhất cho mô hình
    model.load_state_dict(best_model_wts)
    
    # Lưu toàn bộ history ra file JSON để dễ dàng vẽ biểu đồ so sánh các mô hình sau này
    with open('vit_from_scratch_metrics.json', 'w') as f:
        json.dump(history, f, indent=4)
        
    return model, history

In [7]:
print(f"=== BẮT ĐẦU HUẤN LUYỆN VIT FROM SCRATCH")

vit_scratch, history_vit_scratch = train_model_with_metrics(
    model=vit_scratch,
    train_loader=train_loader,      # Sử dụng DataLoader đã tạo (Resize 224x224, 90% train)
    val_loader=val_loader,          # 10% validation clean
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=EPOCHS,
    device=device
)

=== BẮT ĐẦU HUẤN LUYỆN VIT FROM SCRATCH
Epoch 1/50
----------


OutOfMemoryError: CUDA out of memory. Tried to allocate 38.00 MiB. GPU 0 has a total capacity of 7.66 GiB of which 32.81 MiB is free. Including non-PyTorch memory, this process has 7.62 GiB memory in use. Of the allocated memory 7.30 GiB is allocated by PyTorch, and 157.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)